In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes

In [ ]:
# ---------------- 1. KÜTÜPHANELER ----------------
import pandas as pd
import numpy as np
import time
import torch
import gc
import re

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

from huggingface_hub import login, HfApi, snapshot_download, hf_hub_download
from transformers.trainer_utils import get_last_checkpoint

In [ ]:
# ---------------- 2. HUGGING FACE GİRİŞİ ----------------
from google.colab import userdata
hf_token = userdata.get("hf")
login(token=hf_token)

In [ ]:
# ---------------- 3. VERİ SETİ YÜKLEME VE HAZIRLAMA ----------------
print("\nVeri seti Hugging Face Hub üzerinden indiriliyor...")

dataset_path = hf_hub_download(
    repo_id="nihalenc/turkish-8class-emotion-dataset",
    filename="turkish-8class-emotion-dataset.csv",
    repo_type="dataset"
)

df = pd.read_csv(dataset_path, sep=";", encoding="utf-8")
df.columns = df.columns.str.strip()

# --- VERİ TEMİZLİĞİ VE MİNİMAL NORMALİZASYON ---
def clean_text_for_llm(text):
    text = str(text)
    # Linkler model için genelde duygu bilgisini artırmadığı için temizlenir.
    text = re.sub(r"http\S+|www\.\S+", "", text)
    # HTML etiketi varsa silinir; modelin gereksiz sembollerden etkilenmesi azaltılır.
    text = re.sub(r"<.*?>", "", text)
    # Kullanıcı mentionları silinir.
    text = re.sub(r'\@\w+', '', text)
    # Baştaki ve sondaki boşluklar kaldırılır.
    return text.strip()

# Temizleme fonksiyonu tüm metin sütununa uygulanır.
df["text"] = df["text"].apply(clean_text_for_llm)

print(f"Kopya kontrolü öncesi: {len(df)} satır")
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

print(f"Kopya kontrolü sonrası: {len(df)} satır")

emotion_cols = [
    "igrenme",
    "korku",
    "minnet",
    "pismanlik",
    "mutluluk",
    "ofke",
    "saskinlik",
    "uzuntu"
]

label_sum = df[emotion_cols].sum(axis=1)
print("Etiket toplamı dağılımı:")
print(label_sum.value_counts())

assert (label_sum == 1).all(), "Bazı satırlarda 0 veya birden fazla etiket var!"
df["label_name"] = df[emotion_cols].idxmax(axis=1)
label_mapping = {col: idx for idx, col in enumerate(emotion_cols)}
id2label = {idx: col for col, idx in label_mapping.items()}

df["label"] = df["label_name"].map(label_mapping)

print("\nSınıf dağılımı:")
print(df["label_name"].value_counts())

X = df["text"].tolist()
y = df["label"].tolist()

# İlk ayrımda verinin %80'i eğitim, %20'si geçici veri olarak ayrılır.
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Geçici %20 veri ikiye bölünür: %10 validation, %10 test.
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f"\nDağılım -> Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Eğitim Seti (Train): {len(X_train)} satır")
print(f"Doğrulama Seti (Val): {len(X_val)} satır")
print(f"Test Seti (Test): {len(X_test)} satır")

train_dataset = Dataset.from_dict({
    "text": X_train,
    "label": y_train
})

val_dataset = Dataset.from_dict({
    "text": X_val,
    "label": y_val
})

test_dataset = Dataset.from_dict({
    "text": X_test,
    "label": y_test
})

In [ ]:
# ---------------- 4. SINIF AĞIRLIKLARI ----------------
class_weights = compute_class_weight(
    class_weight="balanced",      
    classes=np.unique(y_train),  
    y=y_train                 
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("\nSınıf Ağırlıkları Hesaplandı:")
for emotion, weight in zip(emotion_cols, class_weights):
    print(f"{emotion:<10}: {weight:.4f}")

In [ ]:
# ---------------- 5. MISTRAL MODEL, TOKENIZER, QUANTIZATION VE LORA ----------------

torch.cuda.empty_cache()
gc.collect()

model_id = "mistralai/Mistral-7B-v0.3"

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    add_eos_token=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                 
    bnb_4bit_use_double_quant=True,  
    bnb_4bit_quant_type="nf4",        
    bnb_4bit_compute_dtype=torch.float16  
)

# Mistral modeli sequence classification görevi için yüklenir.
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=8,
    id2label=id2label,
    label2id=label_mapping,
    quantization_config=bnb_config,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.problem_type = "single_label_classification"
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    task_type="SEQ_CLS",    
    r=16,                    
    lora_alpha=32,           
    lora_dropout=0.05,        
    bias="none",             
    target_modules=[
        "q_proj",             
        "k_proj",          
        "v_proj",           
        "o_proj",        
        "gate_proj",         
        "up_proj",          
        "down_proj"          
    ],
    modules_to_save=["score"]
)

# Model PEFT/LoRA formatına dönüştürülür.
model = get_peft_model(model, peft_config)
print("\nTrainable parametreler:")
model.print_trainable_parameters()

In [ ]:
# ---------------- 6. TOKENIZATION ----------------
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,  
        max_length=128    
    )

# Eğitim, validation ve test setleri ayrı ayrı tokenize edilir.
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

In [ ]:
# ---------------- 7. CUSTOM TRAINER VE KALICI KAYIT ----------------
repo_id = "byzhsm/mistral3"
class TamKalicKayitCallback(TrainerCallback):
    def __init__(self, repo_id, token):
        self.repo_id = repo_id
        self.api = HfApi(token=token)

    def on_save(self, args, state, control, **kwargs):
        checkpoint_klasoru = f"{args.output_dir}/checkpoint-{state.global_step}"
        print(f"\n {checkpoint_klasoru} Hugging Face'e kopyalanıyor...")

        try:
            self.api.upload_folder(
                folder_path=checkpoint_klasoru,
                repo_id=self.repo_id,
                path_in_repo=f"checkpoint-{state.global_step}",
                repo_type="model"
            )
            print("Tüm dosyalar Hugging Face'e kaydedildi!")

        except Exception as e:
            print(f" Yüklenemedi: {e}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        labels = labels.to(logits.device)

        # Ağırlıklı CrossEntropyLoss kullanılır.
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights_tensor.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),                                      
        "macro_f1": f1_score(labels, predictions, average="macro"),                          
        "macro_precision": precision_score(labels, predictions, average="macro", zero_division=0), 
        "macro_recall": recall_score(labels, predictions, average="macro", zero_division=0)        
    }

# Eğitim parametreleri.
training_args = TrainingArguments(
    output_dir="./mistral-emotion-results", 

    learning_rate=3e-5,                     
    per_device_train_batch_size=8,         
    per_device_eval_batch_size=16,          
    gradient_accumulation_steps=4,    
          
    gradient_checkpointing=True,           
    num_train_epochs=3,                    
    lr_scheduler_type="cosine",            
    warmup_steps=150,                       

    eval_strategy="steps",                  
    eval_steps=250,                         
    save_strategy="steps",                  
    save_steps=250,                      
    save_total_limit=2,                     

    load_best_model_at_end=True,           
    metric_for_best_model="macro_f1",      
    greater_is_better=True,                

    logging_steps=100,                      

    fp16=True,                             
    optim="paged_adamw_8bit",              

    group_by_length=True,                  
    dataloader_num_workers=2,               

    report_to="none",                       

    push_to_hub=False,                      

    seed=42,                                
    data_seed=42,                           
    weight_decay=0.01,                      
    max_grad_norm=0.3                       
)

# Trainer nesnesi oluşturulur.
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3),    
        TamKalicKayitCallback(repo_id=repo_id, token=hf_token) 
    ]
)

In [ ]:
# ---------------- 8. BULUTTAN KONTROL VE EĞİTİM ----------------

local_output_dir = "./mistral-emotion-results"
api = HfApi(token=hf_token)
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    private=True,
    exist_ok=True
)

print("Hugging Face Hub kontrol ediliyor...")

try:
    snapshot_download(
        repo_id=repo_id,
        repo_type="model",
        local_dir=local_output_dir,
        allow_patterns="checkpoint-*/*",
        token=hf_token
    )

    print("Buluttaki kayıtlar başarıyla yerel diske çekildi!")

except Exception as e:
    # Repo boşsa veya checkpoint yoksa eğitim sıfırdan başlar.
    print("Bulutta eski kayıt bulunamadı veya repo henüz boş. Sıfırdan başlanacak.")

# Yerel klasördeki son checkpoint aranır.
last_checkpoint = get_last_checkpoint(local_output_dir)

# Checkpoint varsa eğitim kaldığı yerden devam eder.
# Yoksa eğitim sıfırdan başlatılır.
if last_checkpoint is not None:
    print(f"Yarım kalmış eğitim bulundu! {last_checkpoint} noktasından devam ediliyor...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Daha önce kayıt bulunamadı, sıfırdan başlanıyor...")
    trainer.train()


# Eğitim sonunda load_best_model_at_end=True sayesinde en iyi macro_f1 checkpoint'i trainer içinde tutulur.
print("Son güncel ağırlıklar Hugging Face Hub'a senkronize ediliyor...")

trainer.save_model(local_output_dir)
tokenizer.save_pretrained(local_output_dir)

try:
    
    api.upload_folder(
        folder_path=local_output_dir,
        repo_id=repo_id,
        commit_message="Mistral duygu sınıflandırma eğitimi ve değerlendirmesi tamamlandı!",
        repo_type="model"
    )

    print("Her şey Hugging Face Hub'a yüklendi!")

except Exception as e:
    print(f"Son yüklemede bir sorun oluştu ancak checkpointler kaydedildi: {e}")

In [ ]:
# ---------------- 9. TEST TAHMİNLERİ, CONFUSION MATRIX VE HATA ANALİZİ ----------------
print("Model test seti üzerinde tahminler yapıyor...")

predictions = trainer.predict(tokenized_test)

# Modelin ürettiği logits değerlerinde en büyük sınıf final tahmin olarak seçilir.
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("\n" + "=" * 70)
print("GENEL TEST METRİKLERİ")
print("=" * 70)
print(f"Test Sonuçları: {predictions.metrics}\n")

print("\n" + "=" * 70)
print("HER DUYGU SINIFI İÇİN BAŞARI YÜZDELERİ")
print("=" * 70)

print(
    classification_report(
        y_true,
        y_pred,
        target_names=emotion_cols,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=emotion_cols,
    yticklabels=emotion_cols
)

plt.xlabel("Modelin Tahmin Ettiği Duygu", fontsize=12, fontweight="bold")
plt.ylabel("Gerçek Duygu", fontsize=12, fontweight="bold")
plt.title("Mistral Duygu Analizi Karmaşıklık Matrisi", fontsize=14, fontweight="bold")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("MODELİN TÜM HATA KOMBİNASYONLARI")
print("=" * 70)

# Hatalı sınıflandırmalar liste halinde tutulur.
hatalar = []

for gercek_idx in range(len(emotion_cols)):
    for tahmin_idx in range(len(emotion_cols)):
        if gercek_idx != tahmin_idx:
            hata_sayisi = cm[gercek_idx, tahmin_idx]

            if hata_sayisi > 0:
                gercek_duygu = emotion_cols[gercek_idx].upper()
                tahmin_duygu = emotion_cols[tahmin_idx].upper()
                hatalar.append((gercek_duygu, tahmin_duygu, hata_sayisi))

# En çok yapılan hata kombinasyonları önce görünecek şekilde sıralanır.
hatalar.sort(key=lambda x: x[2], reverse=True)

for gercek, tahmin, sayi in hatalar:
    print(f"Gerçekte {gercek:<10} olup modelin {tahmin:<10} dediği: {sayi:>3} cümle")

In [ ]:
# ---------------- 10. CLASSIFICATION REPORT'U TABLO VE CSV OLARAK KAYDETME ----------------

report = classification_report(
    y_true,
    y_pred,
    target_names=emotion_cols,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

report_df.to_csv("mistral_classification_report.csv")

In [ ]:
# ---------------- 11. YANLIŞ TAHMİNLERİ CSV OLARAK ÇIKARMA ----------------

test_texts = X_test
error_df = pd.DataFrame({
    "text": test_texts,
    "true_label": [emotion_cols[i] for i in y_true],
    "predicted_label": [emotion_cols[i] for i in y_pred]
})


wrong_df = error_df[error_df["true_label"] != error_df["predicted_label"]]

display(wrong_df.head(30))
wrong_df.to_csv("mistral_wrong_predictions.csv", index=False)
print("Yanlış tahmin sayısı:", len(wrong_df))

In [ ]:
# ---------------- 12. INFERENCE TIME ÖLÇME ----------------

def measure_inference_time(texts, n=100, max_length=128):
    model.eval()

    # Test setinden ilk n örnek alınır.
    sample_texts = texts[:n]

    times = []

    for text in sample_texts:
        # Tek örnek için başlangıç zamanı alınır.
        start = time.time()
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Geçen süre hesaplanır.
        end = time.time()
        times.append(end - start)

    # Ölçülen sürelerin ortalaması alınır.
    avg_time = np.mean(times)
    return avg_time

# İlk 100 test cümlesi üzerinden ortalama inference süresi hesaplanır.
avg_inf_time = measure_inference_time(X_test, n=100, max_length=128)

print(f"Ortalama inference süresi: {avg_inf_time:.4f} saniye/cümle")